<a href="https://colab.research.google.com/github/26b21a4544kietgroup-spec/charan-D/blob/main/Frontend_pym.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
x_supports = np.array([0.0, 10.0, 20.0])
x_loads    = np.array([4.0, 14.0, 15.0])   # change 14.0 to move the truck
P          = np.array([50.0, 120.0, 60.0])

A = np.array([[1, 1, 1],
              list(x_supports),
              [1, -2, 1]], dtype=float)
B = np.array([P.sum(), (P * x_loads).sum(), 0.0])

R = np.linalg.solve(A, B)   # same result as inv(A) @ B
print("det(A) =", round(np.linalg.det(A), 2))
print("R1, R2, R3 =", np.round(R, 2), "kN")

det(A) = 60.0
R1, R2, R3 = [ 52.67  76.67 100.67] kN


In [3]:
CYAN, BLUE, MAG, AMB = '#00f0ff', '#0044ff', '#ff00ff', '#ffaa00'
SC = 0.6      # arrow length per kN
BASE = -40    # projector base height

def line(fig, x, y, z, color, w=3, op=0.8, dash=None):
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', opacity=op,
        line=dict(color=color, width=w, dash=dash) if dash else dict(color=color, width=w),
        hoverinfo='skip', showlegend=False))

def arrow(fig, x, y, z_tail, z_tip, color, text):
    d = np.sign(z_tip - z_tail)
    line(fig, [x, x], [y, y], [z_tail, z_tip], color, 14, 0.25)   # glow
    line(fig, [x, x], [y, y], [z_tail, z_tip], '#ffffff', 3, 0.95) # core
    fig.add_trace(go.Cone(x=[x], y=[y], z=[z_tip], u=[0], v=[0], w=[d],
        anchor='tip', sizemode='absolute', sizeref=7, showscale=False,
        colorscale=[[0, color], [1, color]], hoverinfo='skip'))
    fig.add_trace(go.Scatter3d(x=[x], y=[y], z=[z_tail - 6*d], mode='text',
        text=[text], textfont=dict(color=color, size=13),
        hoverinfo='skip', showlegend=False))

In [4]:
fig = go.Figure()
t = np.linspace(0, 2*np.pi, 100)

# Projector rings + light beams + cone
for r, c, w in [(15, BLUE, 8), (13, '#00aaff', 3), (9, '#00aaff', 2), (5, '#00aaff', 2)]:
    line(fig, 10 + r*np.cos(t), r*np.sin(t), np.full_like(t, BASE), c, w, 0.8)
for a in np.linspace(0, 2*np.pi, 12, endpoint=False):
    line(fig, [10 + 15*np.cos(a), 10], [15*np.sin(a), 0], [BASE, 0], CYAN, 1, 0.2, 'dot')
u = np.linspace(0, 2*np.pi, 48); zc = np.array([BASE, 0.0])
rc = (15 + (2 - 15) * (zc - BASE) / (0 - BASE))[:, None]
fig.add_trace(go.Surface(x=10 + rc*np.cos(u), y=rc*np.sin(u), z=np.repeat(zc[:, None], 48, 1),
    opacity=0.06, showscale=False, colorscale=[[0, '#00aaff'], [1, '#00aaff']], hoverinfo='skip'))

# Deck wireframe
xs, ys, zs = [], [], []
for y in np.linspace(-3, 3, 7):
    xs += [-2, 22, None]; ys += [y, y, None]; zs += [0, 0, None]
for x in np.linspace(-2, 22, 25):
    xs += [x, x, None]; ys += [-3, 3, None]; zs += [0, 0, None]
line(fig, xs, ys, zs, CYAN, 2, 0.45)
line(fig, [-2, 22], [0, 0], [0, 0], '#00ffff', 8, 0.9)   # spine

# Pillars
for x in x_supports:
    line(fig, [x, x], [0, 0], [0, BASE], '#0033ff', 8, 0.7, 'dash')

# Loads (down) and reactions (up)
for x, p in zip(x_loads, P):
    truck = p == 120.0
    arrow(fig, x, 0, 2 + p*SC, 2, MAG if truck else AMB,
          f"TRUCK<br>{p:.0f} kN" if truck else f"{p:.0f} kN")
for i, (x, r) in enumerate(zip(x_supports, R)):
    arrow(fig, x, 3, -2 - r*SC, -2, CYAN, f"R{i+1}<br>{r:.1f} kN")

# HUD + layout
hud = (f"SYS. ONLINE: AX=B PROTOCOL<br>--------------------------<br>"
       f"DET(A)   : {np.linalg.det(A):.2f}<br>TRUCK POS: {x_loads[1]:.1f} m<br>"
       f"MAX LOAD : {P.max():.1f} kN")
fig.add_annotation(text=hud, x=0.02, y=0.85, xref='paper', yref='paper', showarrow=False,
    align='left', font=dict(color='#00ffcc', family='monospace', size=13),
    bgcolor='#001122', bordercolor='#00ffcc', borderpad=10)
fig.update_layout(
    title=dict(text="STRUCTURAL LOAD MATRIX VISUALIZER", x=0.5,
               font=dict(color='white', family='monospace', size=22)),
    paper_bgcolor='black', margin=dict(l=0, r=0, t=60, b=0), height=720, showlegend=False,
    scene=dict(bgcolor='black',
        xaxis=dict(visible=False, range=[-5, 25]),
        yaxis=dict(visible=False, range=[-16, 16]),
        zaxis=dict(visible=False, range=[-45, 125]),
        aspectmode='manual', aspectratio=dict(x=2, y=1.2, z=1.1),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.7))))
fig.show()